In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# 크롬드라이버 경로 설정
DRIVER_PATH = "C:/chromedriver-win64/chromedriver-win64/chromedriver.exe"
service = Service(DRIVER_PATH)
options = Options()
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

# 드라이버 실행
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 20)

# ✅ 기준 스탯 항목 정의 (2016-2017 시즌 고유 스탯 항목)
stat_names = [
    "PK 제외한 xG", "xG 유효 슈팅 (xGOT)", "가로채기", "경고", "공격 지역 점유율",
    "공중 볼 경합 성공", "공중 볼 경합 성공 %", "기회 창출", "드리블 성공", "드리블로 제침",
    "득점", "롱 패스 정확도", "막힘", "반칙", "볼 경합 성공", "볼 경합 성공 %", "볼 뺏김",
    "상대편 박스 내에서의 터치", "성공한 크로스", "성공한 패스", "슛", "어시스트", "예상 골 (xG)",
    "예상 어시스트 (xA)", "유효 슈팅", "정확한 긴 패스", "크로스 정확도", "태클 성공",
    "태클 성공 %", "터치", "퇴장", "패스 정확도", "페널티킥 받음", "페널티킥 허용", "회복",
    "획득한 파울"
]

all_data = []
visited = set()

# ✅ 페이지 반복
page_num = 0
while True:
    page_url = f"https://www.fotmob.com/ko/leagues/47/stats/season/10418/players/rating?page={page_num}"
    driver.get(page_url)
    time.sleep(3)

    try:
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href^='/ko/players/']")))
    except:
        print("더 이상 페이지 없음. 종료합니다.")
        break

    player_links = driver.find_elements(By.CSS_SELECTOR, "a[href^='/ko/players/']")
    unique_links = []

    for link in player_links:
        url = link.get_attribute("href")
        if url not in visited:
            visited.add(url)
            unique_links.append(url)

    for player_url in unique_links:
        try:
            driver.get(player_url)
            wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-zt63wq-PlayerNameCSS")))
            time.sleep(2)

            # 선수명
            try:
                name_el = driver.find_element(By.CLASS_NAME, "css-zt63wq-PlayerNameCSS")
                player_name = name_el.text.strip().lower()
            except:
                player_name = player_url.split("/")[-1].replace("-", " ")

            # 90분당 버튼 클릭
            try:
                buttons = driver.find_elements(By.CLASS_NAME, "css-1efq0w6-FilterButton")
                found_90min = False
                for btn in buttons:
                    if btn.text.strip() == "90분당":
                        btn.click()
                        found_90min = True
                        break
                if not found_90min:
                    print(f"[스킵] 90분당 버튼 없음: {player_url}")
                    continue
            except Exception as e:
                print(f"[예외] 90분당 클릭 중 오류 발생: {player_url} - {e}")
                continue

            time.sleep(3)

            # 스탯 추출
            extracted_stats = {}
            titles = driver.find_elements(By.CLASS_NAME, "css-2duihq-StatTitle")
            values = driver.find_elements(By.CLASS_NAME, "css-jb6lgd-StatValue")

            for title_el, value_el in zip(titles, values):
                label = title_el.text.strip()
                value = value_el.text.strip()
                if label in stat_names:
                    extracted_stats[label] = value

            final_stat = {key: extracted_stats.get(key, '-') for key in stat_names}
            final_stat["선수명"] = player_name
            final_stat["URL"] = player_url
            all_data.append(final_stat)

        except Exception as e:
            print(f"[오류] {player_url}: {e}")
            continue

    print(f"✅ Page {page_num+1} 완료. 수집 선수 수: {len(all_data)}")
    page_num += 1

# ✅ 저장
df = pd.DataFrame(all_data)
df.to_csv("fotmob_2016_2017_all.csv", index=False, encoding="utf-8-sig")
print("✅ 전체 페이지 선수 크롤링 완료 및 CSV 저장! 총 선수 수:", len(df))

# 종료
driver.quit()


[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/248453/paul-pogba
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/156008/jordan-henderson
✅ Page 1 완료. 수집 선수 수: 8
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/363333/dele-alli
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/35724/zlatan-ibrahimovic
✅ Page 2 완료. 수집 선수 수: 16
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/16351/aleksandar-kolarov
✅ Page 3 완료. 수집 선수 수: 25
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/36615/yaya-toure
✅ Page 4 완료. 수집 선수 수: 34
✅ Page 5 완료. 수집 선수 수: 44
✅ Page 6 완료. 수집 선수 수: 54
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/139671/marc-albrighton
✅ Page 7 완료. 수집 선수 수: 63
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/35345/jason-puncheon
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/160496/junior-stanislas
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/47558/joe-allen
✅ Page 8 완료. 수집 선수 수: 70
✅ Page 9 완료. 수집 선수 수: 80
[스킵] 90분당 버튼 없음: https://www.fotmob.com/ko/players/32441